# Text-to-SQL Accuracy Estimation with GLIDE

Spider 1.0 is a large-scale text-to-SQL benchmark covering 200 relational databases across diverse
domains. Each example pairs a natural language question with a gold SQL query.

This case study estimates the accuracy of a Claude Haiku text-to-SQL generator on the Spider
training split. Human evaluation is conceptually straightforward: execute both the gold SQL and
the predicted SQL against the corresponding SQLite database, then compare the result sets.
In practice, running this at scale with human reviewers is expensive.

A cheaper alternative is an LLM judge: given the question, the database schema, and the predicted
SQL, the judge decides whether the translation is correct. The judge is not perfectly calibrated.
Its reliability varies across databases: a simple, well-structured schema is easier to judge
correctly than one with many tables, unusual naming conventions, or domain-specific vocabulary.
Each database therefore forms a natural stratum.

GLIDE applies Stratified PPI++ to correct for the per-database judge bias and produce a valid
confidence interval with far fewer human annotations than a purely classical approach would require.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from datasets import load_dataset

from glide.estimators import (
    ClassicalMeanEstimator,
    StratifiedClassicalMeanEstimator,
    StratifiedPPIMeanEstimator,
)
from glide.samplers import StratifiedSampler
from glide.simulators import simulate_annotation

RANDOM_SEED = 42
BUDGET = 400
CONFIDENCE_LEVEL = 0.95

In [ ]:
HF_REPO = "glide-py/spider-text-to-sql"

dataset = load_dataset(HF_REPO, split="train")

y_true_oracle = np.array(dataset["human_label"], dtype=float)
y_proxy = np.array(dataset["llm_judge_label"], dtype=float)
groups = np.array(dataset["db_id"])

## The Human Annotator

`y_true_oracle` was produced by executing both the gold SQL and the predicted SQL against the
corresponding Spider SQLite database, sorting each result set, and comparing them as Python lists.
An execution error scores 0; an exact result set match scores 1.
This process is deterministic and fully reproducible.

The full dataset and the scripts used to build it are published at
[glide-py/spider-text-to-sql](https://huggingface.co/datasets/glide-py/spider-text-to-sql)
on HuggingFace. In a real deployment, the oracle role would be played by human reviewers
on a sampled subset. Here, the programmatic comparison gives us a complete oracle for all
examples, which we use to simulate the annotation process with `simulate_annotation`.

In [ ]:
df = pd.DataFrame(
    {
        "db_id": groups,
        "human_label": y_true_oracle,
        "llm_judge_label": y_proxy,
    }
)
summary = df.groupby("db_id")[["human_label", "llm_judge_label"]].mean()

x = np.arange(len(summary))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(x - width / 2, summary["human_label"], width, label="Human (ground truth)")
ax.bar(x + width / 2, summary["llm_judge_label"], width, label="LLM Judge (proxy)")
ax.set_xticks(x)
ax.set_xticklabels(summary.index, rotation=45, ha="right")
ax.set_ylabel("Mean accuracy")
ax.legend()
plt.tight_layout()
plt.show()

## Stratified Sampling

Annotating every example with a human reviewer would be expensive. GLIDE draws a fixed budget
using Neyman allocation, which directs more budget to databases where the proxy is least reliable:
databases with higher variance in judge accuracy receive proportionally more of the annotation budget.

In [ ]:
xi = StratifiedSampler().sample(y_proxy, groups, budget=BUDGET, strategy="neyman", random_seed=RANDOM_SEED)
y_true = simulate_annotation(y_true_oracle, xi)
print(f"Annotated: {int(xi.sum())} / {len(xi)} examples")
print(pd.Series(groups[xi == 1]).value_counts().rename("annotated per database").to_string())

In [ ]:
result_judge = ClassicalMeanEstimator().estimate(
    y_proxy,
    metric_name="SQL Accuracy",
    confidence_level=CONFIDENCE_LEVEL,
)
result_human = StratifiedClassicalMeanEstimator().estimate(
    y_true,
    groups,
    metric_name="SQL Accuracy",
    confidence_level=CONFIDENCE_LEVEL,
)
result_glide = StratifiedPPIMeanEstimator().estimate(
    y_true,
    y_proxy,
    groups,
    metric_name="SQL Accuracy",
    confidence_level=CONFIDENCE_LEVEL,
    power_tuning=True,
)
print(result_glide.summary())

In [ ]:
results = [result_judge, result_human, result_glide]
labels = [
    f"LLM Judge ({len(y_proxy)} examples)",
    f"Human sample ({int(xi.sum())} examples)",
    f"GLIDE ({int(xi.sum())} human + {len(y_proxy)} proxy)",
]

fig, ax = plt.subplots(figsize=(8, 3))
for i, (result, label) in enumerate(zip(results, labels)):
    ci = result.confidence_interval
    ax.plot([ci.lower_bound, ci.upper_bound], [i, i], lw=3)
    ax.plot(result.mean, i, "o", ms=7)
    ax.text(ci.upper_bound + 0.005, i, label, va="center", fontsize=9)

ax.axvline(y_true_oracle.mean(), color="black", linestyle="--", lw=1, label="True accuracy")
ax.set_yticks([])
ax.set_xlabel("SQL Accuracy")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## Results

The forest plot illustrates three estimates of text-to-SQL accuracy on Spider 1.0.

**LLM Judge.** The judge estimate uses all proxy labels without bias correction.
The gap between the judge estimate and the true accuracy (dashed line) reveals the direction
and magnitude of the judge's systematic error on this model.

**Human sample.** The stratified classical estimate from the annotated subset is unbiased:
it centers near the true accuracy. Its confidence interval is wide, however, because the
annotated subset is a small fraction of the full dataset.

**GLIDE.** The stratified PPI++ estimate is also unbiased by construction but achieves a
narrower confidence interval by leveraging all proxy labels. The width reduction reflects
the effective sample size gain from combining human annotations with the proxy signal.

The per-database breakdown in the bias chart above reveals why stratification improves over
plain PPI++: databases where the judge is most miscalibrated receive a larger share of the
annotation budget under Neyman allocation, targeting human effort where the proxy is least reliable.